# Use Case 5 — Vision

gpt-4o accepts images as part of the `messages` array.  
Each image is a content block of type `image_url`, carrying either:

- A **public URL** — model fetches it at inference time.
- A **data URL** (`data:[mime];base64,<b64>`) — image embedded inline in the request.

Supported formats: JPEG, PNG, GIF, WEBP (max ~20 MB).

**Prerequisites:** Set `OPENAI_API_KEY` in the kernel environment or a `.env` file.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError("OPENAI_API_KEY is not set")

MODEL = os.environ.get("CHAT_MODEL", "gpt-4o")
print(f"Using model: {MODEL}")

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

## 1. Image from URL

Pass any public image URL directly inside the `image_url` content block.

In [ ]:
# Public domain image — Wikimedia Commons
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"

completion = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": IMAGE_URL, "detail": "auto"},
                },
                {"type": "text", "text": "What is in this image? Describe it."},
            ],
        }
    ],
)

print(completion.choices[0].message.content)
print("\nTokens used:", completion.usage.total_tokens)

## 2. Image from local file (base64)

Read the file, base64-encode it, and send as a `data:` URL.  
The model never needs to reach out to the internet for this path.

In [ ]:
import base64
import mimetypes
from pathlib import Path


def encode_image(path: str) -> tuple[str, str]:
    """Return (mime_type, base64_string) for a local image file."""
    file_path = Path(path)
    mime, _ = mimetypes.guess_type(str(file_path))
    if not mime or not mime.startswith("image/"):
        raise ValueError(f"Cannot determine image MIME type: {file_path}")
    b64 = base64.standard_b64encode(file_path.read_bytes()).decode("utf-8")
    return mime, b64


# Replace with an actual local image path
# image_path = "/path/to/your/image.jpg"
# mime, b64 = encode_image(image_path)
# data_url = f"data:{mime};base64,{b64}"
#
# completion = client.chat.completions.create(
#     model=MODEL,
#     messages=[
#         {
#             "role": "user",
#             "content": [
#                 {
#                     "type": "image_url",
#                     "image_url": {"url": data_url, "detail": "auto"},
#                 },
#                 {"type": "text", "text": "Describe this image."},
#             ],
#         }
#     ],
# )
# print(completion.choices[0].message.content)

print("Set image_path above and uncomment the code to run.")

## 3. Detail levels

The `detail` parameter on `image_url` controls token cost vs accuracy:

| Value | Behaviour |
|-------|----------|
| `auto` | Model picks based on image size (default) |
| `low`  | 85 tokens flat; fast and cheap |
| `high` | 170 tokens per 512×512 tile; accurate for fine detail |

In [ ]:
for detail in ["low", "high"]:
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": IMAGE_URL, "detail": detail},
                    },
                    {"type": "text", "text": "Describe this image in one sentence."},
                ],
            }
        ],
    )
    tokens = completion.usage.total_tokens
    response = completion.choices[0].message.content
    print(f"[detail={detail}] tokens={tokens}")
    print(f"  → {response}\n")